# 🚀 VASUKI Phase 6J — Pilot QLoRA Training Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maniredii/Vasuki-/blob/main/experiments/phase6j/phase6j_training_colab.ipynb)

This notebook runs the official **VASUKI Phase 6J** pilot QLoRA training experiment on a Google Colab T4 GPU (or cloud GPU instance).

### Prerequisites:
1. **Runtime Type:** `Runtime` -> `Change runtime type` -> select **T4 GPU** (Python 3).
2. **Dataset Files:**
   - `phase6j_training_candidate_diversified.jsonl` (593 records)
   - `phase6j_validation.jsonl` (75 records)
   *(Auto-detected if running in local workspace, or prompted to upload if on Colab web)*

**Expected Runtime:** ~8–12 minutes on a free Tesla T4 GPU.

In [15]:
# Step 1: Install training dependencies
import sys
import torch

if torch.cuda.is_available():
    print('GPU detected. Installing Unsloth and optimized QLoRA stack...')
    !pip install -q unsloth
    !pip install -q --no-deps trl peft accelerate bitsandbytes
    print('[OK] GPU library installation complete.')
else:
    print('[!] CPU runtime detected. Installing lightweight dependencies for verification & testing...')
    !pip install -q transformers datasets trl accelerate
    print('[OK] CPU library installation complete.')


[!] CPU runtime detected. Installing lightweight dependencies for verification & testing...
[OK] CPU library installation complete.


In [16]:
# Step 2: Verify GPU availability and VRAM
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available: ", torch.cuda.is_available())

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Device:      {device_name}")
    print(f"Total VRAM:      {vram_gb:.2f} GB")
    print("[OK] GPU environment verified and ready for QLoRA 4-bit fine-tuning.")
    HAS_GPU = True
else:
    print("\n" + "!" * 70)
    print("[!] NOTICE: Running on CPU runtime (CUDA not available).")
    print("    • If in Google Colab: Click 'Runtime' -> 'Change runtime type' -> Select 'T4 GPU'.")
    print("    • If in VS Code / Colab Extension: You can test dataset verification and tokenization below.")
    print("    • Full QLoRA 4-bit training requires a GPU.")
    print("!" * 70)
    HAS_GPU = False

PyTorch Version: 2.11.0+cpu
CUDA Available:  False

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
[!] NOTICE: Running on CPU runtime (CUDA not available).
    • If in Google Colab: Click 'Runtime' -> 'Change runtime type' -> Select 'T4 GPU'.
    • If in VS Code / Colab Extension: You can test dataset verification and tokenization below.
    • Full QLoRA 4-bit training requires a GPU.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


### Step 3: Verify and Load Datasets
This cell automatically locates the datasets locally if in the workspace, or prompts for upload if running in a fresh Colab cloud VM.

In [17]:
# Step 3: Locate, Download (if needed), and Verify Phase 6J Datasets
import os
import hashlib
import json
import urllib.request

VALID_TRAIN_SHA256 = {
    'b5c4c04b91faf4300e4e7f1a77dbd090421a7f89d693228c6be0a63457562e8b',  # Linux / Git LF (Colab default)
    'af9714012101cba1e639bff0b946c78bdeea947f20b341f1803e4352310cc0e2',  # Windows CRLF
}
VALID_VAL_SHA256 = {
    '6e6ad7626955211ed9ec8f4084e668bf9bf240cc2704430642c351e2ba1e48cb',  # Linux / Git LF (Colab default)
    'db816d9bac321deda2aa80dce5552ff994006c42054bac638baf4a5647a4172d',  # Windows CRLF
}

train_filename = 'phase6j_training_candidate_diversified.jsonl'
val_filename = 'phase6j_validation.jsonl'
RAW_BASE_URL = 'https://raw.githubusercontent.com/Maniredii/Vasuki-/main/experiments/phase6j'

def sha256_of_file(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

search_dirs = [
    '.',
    'experiments/phase6j',
    '../experiments/phase6j',
    '/content',
    '/content/Vasuki-/experiments/phase6j',
    'Vasuki-/experiments/phase6j',
    '/content/drive/MyDrive',
    'D:/VASUKI/experiments/phase6j'
]

train_file = None
val_file = None

for d in search_dirs:
    t_path = os.path.join(d, train_filename)
    v_path = os.path.join(d, val_filename)
    if os.path.exists(t_path) and train_file is None:
        if sha256_of_file(t_path) in VALID_TRAIN_SHA256:
            train_file = t_path
    if os.path.exists(v_path) and val_file is None:
        if sha256_of_file(v_path) in VALID_VAL_SHA256:
            val_file = v_path

# Auto-download from GitHub raw if not found locally or if hash invalid
if train_file is None:
    print('[*] Downloading training dataset from GitHub...')
    urllib.request.urlretrieve(f'{RAW_BASE_URL}/{train_filename}', train_filename)
    train_file = train_filename

if val_file is None:
    print('[*] Downloading validation dataset from GitHub...')
    urllib.request.urlretrieve(f'{RAW_BASE_URL}/{val_filename}', val_filename)
    val_file = val_filename

train_hash = sha256_of_file(train_file)
val_hash = sha256_of_file(val_file)

with open(train_file, 'r', encoding='utf-8') as f:
    train_count = sum(1 for line in f if line.strip())
with open(val_file, 'r', encoding='utf-8') as f:
    val_count = sum(1 for line in f if line.strip())

print(f'Training Dataset:   {train_file} ({train_count} records)')
print(f'  SHA-256:          {train_hash} (VALID: {train_hash in VALID_TRAIN_SHA256})')
print(f'Validation Dataset: {val_file} ({val_count} records)')
print(f'  SHA-256:          {val_hash} (VALID: {val_hash in VALID_VAL_SHA256})')

assert train_hash in VALID_TRAIN_SHA256, f'Training dataset hash mismatch: {train_hash}'
assert val_hash in VALID_VAL_SHA256, f'Validation dataset hash mismatch: {val_hash}'
assert train_count == 593, f'Expected 593 training records, got {train_count}'
assert val_count == 75, f'Expected 75 validation records, got {val_count}'
print('\n[OK] Cryptographic integrity and record counts 100% verified!')


Training Dataset:   ./phase6j_training_candidate_diversified.jsonl (593 records)
  SHA-256:          b5c4c04b91faf4300e4e7f1a77dbd090421a7f89d693228c6be0a63457562e8b (VALID: True)
Validation Dataset: /content/Vasuki-/experiments/phase6j/phase6j_validation.jsonl (75 records)
  SHA-256:          6e6ad7626955211ed9ec8f4084e668bf9bf240cc2704430642c351e2ba1e48cb (VALID: True)

[OK] Cryptographic integrity and record counts 100% verified!


In [18]:
# Step 4: Load Base Model and Inject LoRA Adapters
base_model_name = 'unsloth/Qwen2.5-Coder-0.5B'
max_seq_length = 2048

if HAS_GPU:
    from unsloth import FastLanguageModel
    print(f'Loading {base_model_name} with Unsloth 4-bit NF4...')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model_name,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
    print('Injecting LoRA adapters (r=16, alpha=16, dropout=0.05)...')
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        lora_alpha=16,
        lora_dropout=0.05,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=42,
    )
    print('[OK] Model & LoRA initialized successfully on GPU.')
else:
    from transformers import AutoTokenizer
    print(f'CPU Mode: Loading tokenizer for {base_model_name}...')
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = None
    print('[OK] Tokenizer loaded successfully for validation and testing.')


CPU Mode: Loading tokenizer for unsloth/Qwen2.5-Coder-0.5B...
[OK] Tokenizer loaded successfully for validation and testing.


In [19]:
# Step 5: Format Datasets with Alpaca Template and Verify Token Lengths
from datasets import Dataset
import torch

# Resilient DataCollator import with universal self-contained fallback
try:
    from trl import DataCollatorForCompletionOnlyLM
except ImportError:
    try:
        from trl.trainer import DataCollatorForCompletionOnlyLM
    except ImportError:
        from transformers import DataCollatorForLanguageModeling

        class DataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
            """Self-contained completion loss collator that masks prompt tokens with -100."""
            def __init__(self, response_template, tokenizer, mlm=False, **kwargs):
                super().__init__(tokenizer=tokenizer, mlm=mlm, **kwargs)
                self.response_template = response_template
                if isinstance(response_template, str):
                    self.response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)
                else:
                    self.response_token_ids = response_template

            def torch_call(self, examples):
                batch = super().torch_call(examples)
                labels = batch["labels"].clone()
                rlen = len(self.response_token_ids)
                for i in range(len(examples)):
                    input_ids = batch["input_ids"][i].tolist()
                    response_start = None
                    for j in range(len(input_ids) - rlen + 1):
                        if input_ids[j:j+rlen] == self.response_token_ids:
                            response_start = j + rlen
                            break
                    if response_start is not None:
                        labels[i, :response_start] = -100
                    else:
                        labels[i, :] = -100
                batch["labels"] = labels
                return batch

def load_and_format(filepath):
    records = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    formatted = []
    for r in records:
        prompt = f"### Instruction:\n{r['instruction']}\n\n"
        if r.get('input'):
            prompt += f"### Input:\n{r['input']}\n\n"
        prompt += "### Response:\n"
        formatted.append({"text": prompt + r['response']})
    return Dataset.from_list(formatted), records

train_dataset, train_raw = load_and_format(train_file)
val_dataset, val_raw = load_and_format(val_file)

# Calculate actual token lengths with Qwen tokenizer
lengths = [len(tokenizer.encode(item["text"])) for item in train_dataset]
print(f"Training set token stats: Min={min(lengths)}, Mean={sum(lengths)/len(lengths):.1f}, Max={max(lengths)} tokens")
print(f"Max configured sequence length: {max_seq_length} tokens")
print(f"Truncation risk: 0.0% (Largest record is {max(lengths)} tokens, well within {max_seq_length})")

# Setup response completion collator (masks instruction tokens with -100)
response_template = "### Response:\n"
collator = DataCollatorForCompletionOnlyLM(response_template=response_template, tokenizer=tokenizer)
print("\n[OK] Datasets formatted and completion loss collator configured.")


Training set token stats: Min=30, Mean=136.1, Max=381 tokens
Max configured sequence length: 2048 tokens
Truncation risk: 0.0% (Largest record is 381 tokens, well within 2048)

[OK] Datasets formatted and completion loss collator configured.


In [23]:
# Step 6: Configure and Run the 220-Step Pilot QLoRA Training
if HAS_GPU and model is not None:
    from transformers import TrainingArguments
    from trl import SFTTrainer
    import time

    output_dir = './phase6j_output'

    common_kwargs = dict(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        max_steps=220,
        learning_rate=2e-4,
        fp16=True,
        bf16=False,
        logging_steps=10,
        eval_steps=50,
        save_steps=50,
        save_total_limit=2,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir=output_dir,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        report_to='none'
    )
    try:
        training_args = TrainingArguments(eval_strategy='steps', **common_kwargs)
    except TypeError:
        training_args = TrainingArguments(evaluation_strategy='steps', **common_kwargs)

    trainer_kwargs = dict(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        dataset_text_field='text',
        data_collator=collator,
        args=training_args,
    )
    try:
        trainer = SFTTrainer(max_seq_length=max_seq_length, **trainer_kwargs)
    except TypeError:
        trainer = SFTTrainer(**trainer_kwargs)

    print('Starting training run (220 steps, ~3 epochs)...')
    start_time = time.time()
    train_stats = trainer.train()
    duration = time.time() - start_time

    print(f'\n[OK] Training complete in {duration/60:.2f} minutes!')
    print(f'    Final Training Loss: {train_stats.training_loss:.4f}')

    eval_stats = trainer.evaluate()
    print(f'    Validation Loss:     {eval_stats["eval_loss"]:.4f}')
else:
    print('\n[!] Training skipped: GPU is required for full QLoRA fine-tuning.')
    print('    To train: Switch runtime to T4 GPU in Google Colab, or run on a cloud GPU instance.')



[!] Training skipped: GPU is required for full QLoRA fine-tuning.
    To train: Switch runtime to T4 GPU in Google Colab, or run on a cloud GPU instance.


In [21]:
# Step 7: Test the Trained Model on Key Spot-Check Prompts
if HAS_GPU and model is not None:
    FastLanguageModel.for_inference(model)

    test_prompts = [
        "Explain Python list comprehensions with a simple example.",
        "How can I call a Java REST API from Python?",
        "Write a complete C++ game engine with OpenGL.",
        "Implement binary search on a sorted list in Python."
    ]

    for prompt in test_prompts:
        formatted = f"### Instruction:\n{prompt}\n\n### Response:\n"
        inputs = tokenizer(formatted, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.2, top_p=0.9)
        resp_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        print("=" * 60)
        print("Prompt:", prompt)
        print("Response:")
        print(resp_text.strip())
        print("=" * 60)
else:
    print("Inference test skipped on CPU.")

Inference test skipped on CPU.


In [22]:
# Step 8: Export Adapter, Merged Model, and Download Zip Package
if HAS_GPU and model is not None:
    import shutil

    # 1. Save LoRA adapter
    adapter_dir = "./phase6j_output/adapter"
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"Saved adapter to {adapter_dir}")

    # 2. Save Merged 16-bit and GGUF
    merged_dir = "./phase6j_output/merged_16bit"
    model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
    print(f"Saved merged 16-bit model to {merged_dir}")

    # 3. Export to GGUF (Q4_K_M)
    try:
        model.save_pretrained_gguf(merged_dir, tokenizer, quantization_method="q4_k_m")
        print("Exported GGUF.")
    except Exception as e:
        print(f"GGUF export note: {e}")

    # 4. Zip output for download
    shutil.make_archive("vasuki_phase6j_output", "zip", "./phase6j_output")
    print("Archive created: vasuki_phase6j_output.zip")

    # Trigger download if running in Colab web interface
    try:
        from google.colab import files
        files.download("vasuki_phase6j_output.zip")
    except ImportError:
        print("Running outside Colab web; archive saved to ./vasuki_phase6j_output.zip")
else:
    print("Artifact export skipped on CPU.")

Artifact export skipped on CPU.
